project setup

In [1]:
!pip install transformers datasets rouge_score evaluate accelerate sentencepiece -q


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


DownLoad  dataset

In [3]:
!pip install kaggle -q
# upload kaggle.json (from your Kaggle account settings) first
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
!kaggle datasets download -d pariza/bbc-news-summary
!unzip -q bbc-news-summary.zip -d bbc_data

Dataset URL: https://www.kaggle.com/datasets/pariza/bbc-news-summary
License(s): CC0-1.0
100% 8.91M/8.91M [00:00<00:00, 52.3MB/s]



Build a DataFrame (article, summary pairs)

In [4]:
import os, pandas as pd

base = "bbc_data/BBC News Summary"
categories = os.listdir(f"{base}/News Articles")

rows = []
for cat in categories:
    art_dir = f"{base}/News Articles/{cat}"
    sum_dir = f"{base}/Summaries/{cat}"
    for fname in os.listdir(art_dir):
        try:
            with open(f"{art_dir}/{fname}", encoding="latin-1") as f:
                article = f.read().strip()
            with open(f"{sum_dir}/{fname}", encoding="latin-1") as f:
                summary = f.read().strip()
            rows.append({"category": cat, "article": article, "summary": summary})
        except Exception as e:
            print(fname, e)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(2225, 3)


,category,article,summary
0,sport,Bellamy under new fire\n\nNewcastle boss Graem...,Souness - who refused to refer to the 25-year-...
1,sport,Ferrero eyes return to top form\n\nFormer worl...,"""It was difficult because I had been playing w..."
2,sport,Unclear future for striker Baros\n\nLiverpool ...,"He told Czech newspaper Daily Sport: ""I don't ..."
3,sport,O'Sullivan quick to hail Italians\n\nIreland c...,"""It was a hell of a tough game,"" said O'Sulliv..."
4,sport,Charvis set to lose fitness bid\n\nFlanker Col...,Flanker Colin Charvis is unlikely to play any ...


In [5]:
df.isnull().sum()

,0
category,0
article,0
summary,0


Preprocessing  & Train/val/test split

In [6]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# 1. Text cleaning
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\.(?=[A-Z])', '. ', text)  # insert space after sentence-ending periods
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['article'] = df['article'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)

print("After cleaning:", df.shape)

# 2. Remove empty / too-short rows
df = df[df['article'].str.len() > 0]
df = df[df['summary'].str.len() > 0]


df = df[df['article'].str.split().str.len() > 20]
df = df[df['summary'].str.split().str.len() > 5]

df = df.reset_index(drop=True)
print("After removing empty/short rows:", df.shape)

#  3. Remove duplicates
before = df.shape[0]
df = df.drop_duplicates(subset=['article'])
df = df.drop_duplicates(subset=['article', 'summary'])
df = df.reset_index(drop=True)
print(f"Removed {before - df.shape[0]} duplicate rows. New shape:", df.shape)

#  4. Check for nulls
print(df.isnull().sum())
df = df.dropna(subset=['article', 'summary']).reset_index(drop=True)

#  5. Train / Val / Test split
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['category']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['category']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

#  6. Sanity check
print(train_df.head(2))
print(train_df['category'].value_counts())

After cleaning: (2225, 3)
After removing empty/short rows: (2225, 3)
Removed 100 duplicate rows. New shape: (2125, 3)
category    0
article     0
summary     0
dtype: int64
Train: (1700, 3)
Val:   (212, 3)
Test:  (213, 3)
   category                                            article  \
0  politics  Security papers 'found in street' An inquiry i...   
1  politics  Guantanamo four free in weeks All four Britons...   

                                             summary  
0  The police spokesman said the newspaper handed...  
1  Civil rights group Liberty said it was "deligh...  
category
business         403
sport            402
politics         322
entertainment    295
tech             278
Name: count, dtype: int64


In [7]:
print(len(train_df), len(val_df), len(test_df))

1700 212 213


Choose a model

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Tokenize as HF Dataset

In [9]:
from datasets import Dataset

prefix = "summarize: "

ARTICLE_MAX_LEN = 768
SUMMARY_MAX_LEN = 128

def preprocess(examples):
    inputs = [prefix + a for a in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=ARTICLE_MAX_LEN,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=SUMMARY_MAX_LEN,
        truncation=True,
        padding="max_length"
    )
    labels["input_ids"] = [
        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_ds = Dataset.from_pandas(train_df[["article","summary"]])
val_ds = Dataset.from_pandas(val_df[["article","summary"]])
test_ds = Dataset.from_pandas(test_df[["article","summary"]])

train_ds = train_ds.map(preprocess, batched=True, remove_columns=["article","summary"])
val_ds = val_ds.map(preprocess, batched=True, remove_columns=["article","summary"])
test_ds = test_ds.map(preprocess, batched=True, remove_columns=["article","summary"])

Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Map:   0%|          | 0/213 [00:00<?, ? examples/s]

Training loop

In [10]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate, numpy as np

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/bbc_summarizer_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=8,
    predict_with_generate=True,
    generation_max_length=128,     # <-- fixes the bug
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    save_total_limit=2,
)

from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.331345,0.230418,0.549700,0.409600,0.405300,0.405100
2,0.294696,0.207008,0.583000,0.447900,0.446800,0.447100
3,0.270376,0.199372,0.602900,0.472600,0.461900,0.462300
4,0.233110,0.193401,0.632400,0.511300,0.486700,0.487000
5,0.232041,0.189007,0.656500,0.543700,0.499600,0.500200
6,0.235153,0.184906,0.659500,0.547600,0.506400,0.507000
7,0.205799,0.184574,0.665100,0.552000,0.507900,0.508400
8,0.220212,0.184522,0.665500,0.554100,0.510400,0.510900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1704, training_loss=0.28113884461317823, metrics={'train_runtime': 2739.7889, 'train_samples_per_second': 4.964, 'train_steps_per_second': 0.622, 'total_flos': 1.2422740967424e+16, 'train_loss': 0.28113884461317823, 'epoch': 8.0})

Evaluate on test set

In [11]:
results = trainer.evaluate(test_ds)

print("Test Results")
print("----------------")

for key, value in results.items():
    print(key, ":", value)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
0.220212,0.177568,8,0.649500,0.537400,0.489500,0.489700


Test Results
----------------
eval_loss : 0.17756816744804382
eval_rouge1 : 0.6495
eval_rouge2 : 0.5374
eval_rougeL : 0.4895
eval_rougeLsum : 0.4897


Inference

In [12]:
def summarize(text, max_length=100, min_length=25, num_beams=5):

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(trainer.model.device)

    outputs = trainer.model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        early_stopping=True
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

for i in range(5):

    article = test_df.iloc[i]["article"]
    actual_summary = test_df.iloc[i]["summary"]

    generated_summary = summarize(article)

    print("=" * 80)
    print("ARTICLE:")
    print(article[:1000])

    print("\nREAL SUMMARY:")
    print(actual_summary)

    print("\nGENERATED SUMMARY:")
    print(generated_summary)

ARTICLE:
Bank holds interest rate at 4.75% The Bank of England has left interest rates on hold again at 4.75%, in a widely-predicted move. Rates went up five times from November 2003 - as the bank sought to cool the housing market and consumer debt - but have remained unchanged since August. Recent data has indicated a slowdown in manufacturing and consumer spending, as well as in mortgage approvals. And retail sales disappointed over Christmas, with analysts putting the drop down to less consumer confidence. Rising interest rates and the accompanying slowdown in the housing market have knocked consumers' optimism, causing a sharp fall in demand for expensive goods, according to a report earlier this week from the British Retail Consortium. The BRC said Britain's retailers had endured their worst Christmas in a decade. "Today's no change decision is correct," said David Frost, Director General of the British Chambers of Commerce (BCC). "But, if there are clear signs that the economy sl

Evakuation

In [13]:
import evaluate

rouge = evaluate.load("rouge")

def check_summary(article, real_summary):

    generated_summary = summarize(article)

    scores = rouge.compute(
        predictions=[generated_summary],
        references=[real_summary],
        use_stemmer=True
    )

    print("GENERATED SUMMARY:")
    print(generated_summary)

    print("\nREAL SUMMARY:")
    print(real_summary)

    print("\nROUGE SCORES:")

    for key, value in scores.items():
        print(f"{key}: {value:.4f}")

    return generated_summary, scores

In [14]:
article = test_df.iloc[0]["article"]
real_summary = test_df.iloc[0]["summary"]

check_summary(article, real_summary)

GENERATED SUMMARY:
CBI chief economist Ian McCafferty said the economy had "slowed in recent months in response to rate rises" but that it was difficult to gauge from the Christmas period the likely pace of activity through the summer. Rising interest rates and the accompanying slowdown in the housing market have knocked consumers' optimism, causing a sharp fall in demand for expensive goods, according to a report earlier this week from the British Retail Consortium. The ONS said manufacturing

REAL SUMMARY:
CBI chief economist Ian McCafferty said the economy had "slowed in recent months in response to rate rises" but that it was difficult to gauge from the Christmas period the likely pace of activity through the summer. Manufacturers' organisation, the EEF, said it expected the hold in interest rates to continue in the near future."The Bank is having to juggle the emergence of inflationary pressures, driven by a tight labour market and buoyant commodity prices, against the risk of an 

('CBI chief economist Ian McCafferty said the economy had "slowed in recent months in response to rate rises" but that it was difficult to gauge from the Christmas period the likely pace of activity through the summer. Rising interest rates and the accompanying slowdown in the housing market have knocked consumers\' optimism, causing a sharp fall in demand for expensive goods, according to a report earlier this week from the British Retail Consortium. The ONS said manufacturing',
 {'rouge1': np.float64(0.49190938511326865),
  'rouge2': np.float64(0.46254071661237783),
  'rougeL': np.float64(0.47249190938511326),
  'rougeLsum': np.float64(0.47249190938511326)})

In [15]:
results = []

for i in range(10):

    article = test_df.iloc[i]["article"]
    reference = test_df.iloc[i]["summary"]

    prediction = summarize(article)

    score = rouge.compute(
        predictions=[prediction],
        references=[reference],
        use_stemmer=True
    )

    results.append({
        "index": i,
        "ROUGE-1": score["rouge1"],
        "ROUGE-2": score["rouge2"],
        "ROUGE-L": score["rougeL"],
        "Reference": reference,
        "Generated": prediction
    })

results_df = pd.DataFrame(results)

results_df

,index,ROUGE-1,ROUGE-2,ROUGE-L,Reference,Generated
0,0,0.491909,0.462541,0.472492,CBI chief economist Ian McCafferty said the ec...,CBI chief economist Ian McCafferty said the ec...
1,1,0.522059,0.437037,0.470588,"""We controlled the game in the first half but ...",Hodgson failed to convert three penalties and ...
2,2,0.491103,0.387097,0.412811,Mr Blair has probably been closer to President...,Mr Blair has probably been closer to President...
3,3,0.297619,0.179641,0.238095,"Healey, twice a Heineken Cup winner, believes ...",Leicester wing Austin Healey hopes to use Sund...
4,4,0.656000,0.653226,0.656000,The Reserve Bank of Australia lifted interest ...,The Reserve Bank of Australia lifted interest ...
5,5,0.779487,0.746114,0.471795,Ms Bradburn said there was a big queue at the ...,She said the people that had bought the famous...
6,6,0.647343,0.585366,0.376812,Lib Dem leader Charles Kennedy has said voters...,Mr Kennedy said during his nearly 22 years in ...
7,7,0.685393,0.579545,0.550562,"Mortgage lending rose by 7.1bn in December, up...",The number of mortgages approved in the UK has...
8,8,0.462462,0.447130,0.312312,"""After that Downing Street proceeded to set ou...",The government has consistently refused to pub...
9,9,0.760563,0.657143,0.464789,"Nicholas Betts-Green, who had been selected to...",A prospective candidate for the UK Independenc...


Predict / generate summaries

In [21]:
def summarize(text, max_length=100, min_length=25, num_beams=5):

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(trainer.model.device)

    outputs = trainer.model.generate(
        **inputs,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=1.0,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        early_stopping=True
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

user_text = input("Paste your article text here:\n")
print("\nARTICLE:\n", user_text[:500], "...\n")
print("MODEL SUMMARY:\n", summarize(user_text))

Paste your article text here:
Elvis regains top chart position  Elvis Presley has scored his 19th number one single in the UK charts with the re-release of Jailhouse Rock, 27 years after his death.  Elvis knocked X Factor winner Steve Brookstein down into second place after three weeks in the charts. In at number three was Iron Maiden for the Number Of The Beast and Erasure entered the chart at four with Breathe. Elvis's number one is the 999th in chart history and comes the day after what would have been his 70th birthday. Fans around the world held tribute events for the singer on Saturday, ranging from concerts to memorabilia exhibitions. Meanwhile, a poll carried out by royalty payments group the Performing Right Society found that The Wonder of You is the Elvis song most performed by live bands and tribute acts.  Record company SonyBMG are releasing Elvis's 18 number one singles at the rate of one a week in Britain, complete with original artwork and a collector's box. Hit single 

save the best model

In [17]:
save_path = "/content/drive/MyDrive/bbc_summarizer_best_01"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/bbc_summarizer_best_01


In [22]:
import os

save_path = "/content/drive/MyDrive/bbc_summarizer_best_01"
file_path = f"{save_path}/model.safetensors"

size_bytes = os.path.getsize(file_path)
print(f"{size_bytes:,} bytes")
print(f"{size_bytes / (1024**2):.2f} MB")
print(f"{size_bytes / (1024**3):.2f} GB")

891,644,712 bytes
850.34 MB
0.83 GB


In [25]:
import shutil, os

save_path = "/content/drive/MyDrive/bbc_summarizer_best_01"
dest_folder = "/content/drive/MyDrive/summarizer_model_files"

os.makedirs(dest_folder, exist_ok=True)

files_needed = [
    "config.json",
    "generation_config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

for f in files_needed:
    src = f"{save_path}/{f}"
    dst = f"{dest_folder}/{f}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied: {f}")
    else:
        print(f"Missing: {f}")

print("\nDone. Files are in Google Drive at:", dest_folder)

Missing: config.json
Missing: generation_config.json
Missing: model.safetensors
Missing: tokenizer.json
Missing: tokenizer_config.json

Done. Files are in Google Drive at: /content/drive/MyDrive/summarizer_model_files
